# Evaluación del sistema web (OE4): usabilidad y rendimiento

Corresponde al **objetivo específico 4** y a la **HE4**: el sistema alcanza un puntaje SUS ≥ 68 con participantes de Lima Metropolitana y un tiempo de respuesta menor a 2 segundos.

- **Usabilidad:** escala SUS en su versión en español validada (Sevilla-Gonzalez et al., 2020). Puntaje por participante según Brooke (1996): ítems impares, respuesta − 1; ítems pares, 5 − respuesta; suma × 2.5 (0 a 100).
- **Rendimiento:** tiempo que tarda el servidor en calcular cada estimación (se registra en cada evaluación). No incluye la espera de la red.
- **Funcionalidad:** el sistema registra cada evaluación y cada encuesta; aquí se verifica la integridad de los registros. Las pruebas automáticas del backend se reportan aparte.

Los datos se descargan con `evaluacion_usabilidad/descargar_resultados.py` y se guardan en `data/usabilidad/` (no se versionan). Aquí solo se exportan tablas **agregadas**.

In [1]:
import ast
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option('display.float_format', '{:.2f}'.format)
DATOS = Path(r'../../data/usabilidad')
SALIDA = Path(r'../../resultados/usabilidad')
SUS_JS = Path(r'../../frontend/js/sus.js')
SALIDA.mkdir(parents=True, exist_ok=True)
META_SUS, META_MS = 68, 2000

sus = pd.read_csv(DATOS / 'sus.csv')
# El archivo de evaluaciones (tiempo de respuesta) puede llegar después que el de la SUS
HAY_EV = (DATOS / 'evaluaciones.csv').exists()
ev = pd.read_csv(DATOS / 'evaluaciones.csv') if HAY_EV else None
sus['respuestas'] = sus['respuestas'].apply(lambda v: ast.literal_eval(v) if isinstance(v, str) else v)
print(f'Encuestas SUS registradas: {len(sus)} | evaluaciones registradas: {len(ev) if HAY_EV else "archivo no disponible"}')

Encuestas SUS registradas: 15 | evaluaciones registradas: archivo no disponible


## 1. Integridad de los registros

In [2]:
def puntaje_sus(r):
    return 2.5 * sum((x - 1) if i % 2 == 0 else (5 - x) for i, x in enumerate(r))


comprobaciones = {
    'Encuestas con 10 respuestas': bool(sus['respuestas'].apply(len).eq(10).all()),
    'Respuestas dentro de la escala 1-5': bool(sus['respuestas'].apply(lambda r: all(1 <= x <= 5 for x in r)).all()),
    'Puntaje guardado = puntaje recalculado': bool(np.allclose(sus['respuestas'].apply(puntaje_sus), sus['puntaje_sus'])),
    'Un solo registro SUS por participante': bool(sus['participante_codigo'].is_unique),
    'Códigos con formato P## (dos dígitos)': bool(sus['participante_codigo'].str.fullmatch(r'P\d{2}').all()),
}
for k, v in comprobaciones.items():
    print(('OK   ' if v else 'REVISAR ') + k)
duplicados = sus[sus['participante_codigo'].duplicated(keep=False)].sort_values('participante_codigo')
if len(duplicados):
    print('\nCódigos con más de una encuesta (revisar antes de analizar):')
    display(duplicados[['id', 'participante_codigo', 'puntaje_sus']])

if HAY_EV:
    con_codigo = ev['participante_codigo'].notna() & (ev['participante_codigo'].astype(str).str.strip() != '')
    participantes_ev = set(ev.loc[con_codigo, 'participante_codigo'])
    participantes_sus = set(sus['participante_codigo'])
    integridad = pd.DataFrame([
        ['Participantes con encuesta SUS', len(participantes_sus)],
        ['Participantes con al menos una evaluación (con código)', len(participantes_ev)],
        ['Participantes con evaluación y encuesta', len(participantes_ev & participantes_sus)],
        ['Con encuesta pero sin evaluación vinculada', len(participantes_sus - participantes_ev)],
        ['Con evaluación pero sin encuesta', len(participantes_ev - participantes_sus)],
        ['Evaluaciones sin código de participante', int((~con_codigo).sum())],
        ['Evaluaciones registradas', len(ev)],
    ], columns=['Indicador', 'Valor'])
    integridad.to_csv(SALIDA / 'integridad_registros.csv', index=False)
    display(integridad)
else:
    print('Sin el archivo de evaluaciones no se puede verificar el vínculo entre evaluaciones y encuestas.')

OK   Encuestas con 10 respuestas
OK   Respuestas dentro de la escala 1-5
OK   Puntaje guardado = puntaje recalculado
OK   Un solo registro SUS por participante
REVISAR Códigos con formato P## (dos dígitos)
Sin el archivo de evaluaciones no se puede verificar el vínculo entre evaluaciones y encuestas.


## 2. Participantes de la evaluación

In [3]:
if HAY_EV:
    u = ev[ev['participante_codigo'].notna()].sort_values('id').drop_duplicates('participante_codigo')
    muestra = pd.DataFrame([
        ['Participantes (con evaluación vinculada)', len(u)],
        ['Edad, media (DE)', f"{u['edad'].mean():.1f} ({u['edad'].std():.1f})"],
        ['Edad, rango', f"{int(u['edad'].min())}–{int(u['edad'].max())}"],
        ['Mujeres, n (%)', f"{int((u['sexo'] == 'femenino').sum())} ({100 * (u['sexo'] == 'femenino').mean():.0f} %)"],
        ['IMC, media (DE)', f"{u['imc'].mean():.1f} ({u['imc'].std():.1f})"],
    ], columns=['Característica', 'Valor'])
    muestra.to_csv(SALIDA / 'participantes.csv', index=False)
    display(muestra)
else:
    print('Sin el archivo de evaluaciones no se pueden describir los participantes.')

Sin el archivo de evaluaciones no se pueden describir los participantes.


## 3. Usabilidad: puntaje SUS

Con 15 participantes se reportan la media con su intervalo de confianza (distribución t), la mediana y la proporción de participantes con puntaje ≥ 68. Para contrastar la meta se usa la prueba t de una muestra frente a 68 (unilateral) y, como comprobación por el tamaño de la muestra, la prueba de Wilcoxon de rangos con signo.

In [4]:
x = sus['puntaje_sus'].to_numpy(dtype=float)
n = len(x)
media, de = x.mean(), x.std(ddof=1)
t_crit = stats.t.ppf(0.975, n - 1)
li, ls = media - t_crit * de / np.sqrt(n), media + t_crit * de / np.sqrt(n)
t_stat, p_t = stats.ttest_1samp(x, META_SUS, alternative='greater')
w_stat, p_w = stats.wilcoxon(x - META_SUS, alternative='greater') if np.any(x != META_SUS) else (np.nan, np.nan)


def interpretacion(p):
    # Mismos puntos de corte que muestra el sistema al participante
    return 'Excelente' if p >= 80.3 else 'Buena' if p >= 68 else 'Aceptable' if p >= 51 else 'Deficiente'


resumen_sus = pd.DataFrame([
    ['Participantes (n)', n],
    ['Puntaje SUS, media', media],
    ['Desviación estándar', de],
    ['IC 95 % de la media (inferior)', li],
    ['IC 95 % de la media (superior)', ls],
    ['Mediana', np.median(x)],
    ['Mínimo', x.min()],
    ['Máximo', x.max()],
    [f'Participantes con puntaje ≥ {META_SUS}, n', int((x >= META_SUS).sum())],
    [f'Participantes con puntaje ≥ {META_SUS}, %', 100 * (x >= META_SUS).mean()],
    ['Prueba t de una muestra frente a 68 (unilateral): p', f'{p_t:.2e}'],
    ['Prueba de Wilcoxon frente a 68 (unilateral): p', f'{p_w:.2e}'],
    ['Interpretación de la media', interpretacion(media)],
], columns=['Indicador', 'Valor'])
resumen_sus.to_csv(SALIDA / 'sus_resumen.csv', index=False)
categorias = pd.Series([interpretacion(v) for v in x]).value_counts().reindex(['Excelente', 'Buena', 'Aceptable', 'Deficiente'], fill_value=0)
categorias.rename('participantes').to_frame().to_csv(SALIDA / 'sus_categorias.csv')
display(resumen_sus)
categorias.rename('participantes').to_frame()

,Indicador,Valor
0,Participantes (n),15
1,"Puntaje SUS, media",93.83
2,Desviación estándar,7.13
3,IC 95 % de la media (inferior),89.89
4,IC 95 % de la media (superior),97.78
5,Mediana,95.00
6,Mínimo,77.50
7,Máximo,100.00
8,"Participantes con puntaje ≥ 68, n",15
9,"Participantes con puntaje ≥ 68, %",100.00


,participantes
Excelente,13
Buena,2
Aceptable,0
Deficiente,0


### Puntaje por ítem

Contribución media de cada ítem al puntaje (0 a 4; mayor es mejor, ya invertidos los ítems pares). Permite ver qué aspectos de la usabilidad fueron más débiles.

In [5]:
items = re.findall(r"'([^']+)',", re.search(r'PREGUNTAS_SUS = \[(.*?)\];', SUS_JS.read_text(encoding='utf-8'), re.S).group(1))
assert len(items) == 10
contrib = pd.DataFrame([[(v - 1) if i % 2 == 0 else (5 - v) for i, v in enumerate(r)] for r in sus['respuestas']])
por_item = pd.DataFrame({
    'ítem': range(1, 11),
    'enunciado': [t.strip() for t in items],
    'contribución media (0-4)': contrib.mean().to_numpy(),
    'DE': contrib.std(ddof=1).to_numpy(),
    'respuesta media (1-5)': pd.DataFrame(sus['respuestas'].tolist()).mean().to_numpy(),
})
por_item.to_csv(SALIDA / 'sus_por_item.csv', index=False)
por_item.sort_values('contribución media (0-4)')

,ítem,enunciado,contribución media (0-4),DE,respuesta media (1-5)
8,9,Me sentí muy confiado al usar la herramienta,3.47,0.83,4.47
1,2,Considero que esta herramienta es innecesariam...,3.73,0.46,1.27
6,7,Imagino que la mayoría de las personas aprende...,3.73,0.46,4.73
7,8,Considero que el uso de esta herramienta es te...,3.73,0.46,1.27
9,10,Necesité saber bastantes cosas antes de poder ...,3.73,0.46,1.27
2,3,Considero que la herramienta es fácil de usar.,3.80,0.41,4.80
3,4,Considero necesario el apoyo de personal exper...,3.80,0.41,1.20
0,1,Me gustaría usar esta herramienta frecuentemente.,3.80,0.41,4.80
4,5,Considero que las funciones de la herramienta ...,3.87,0.35,4.87
5,6,Considero que la herramienta presenta muchas c...,3.87,0.35,1.13


## 4. Rendimiento: tiempo de respuesta

Tiempo que tarda el servidor en calcular cada estimación. Se analizan todas las evaluaciones registradas.

In [6]:
if HAY_EV:
    ms = ev['tiempo_prediccion_ms'].to_numpy(dtype=float)
    rend = pd.DataFrame([
        ['Evaluaciones analizadas', len(ms)],
        ['Tiempo medio (ms)', ms.mean()],
        ['Desviación estándar (ms)', ms.std(ddof=1) if len(ms) > 1 else np.nan],
        ['Mediana (ms)', np.median(ms)],
        ['Percentil 95 (ms)', np.percentile(ms, 95)],
        ['Máximo (ms)', ms.max()],
        [f'Evaluaciones con tiempo < {META_MS} ms, %', 100 * (ms < META_MS).mean()],
    ], columns=['Indicador', 'Valor'])
    rend.to_csv(SALIDA / 'rendimiento.csv', index=False)
    display(rend)
else:
    print('Pendiente: falta el archivo de evaluaciones (data/usabilidad/evaluaciones.csv) para analizar el tiempo de respuesta.')

Pendiente: falta el archivo de evaluaciones (data/usabilidad/evaluaciones.csv) para analizar el tiempo de respuesta.


## 5. Contrastación de la HE4

In [7]:
cumple_sus = media >= META_SUS
filas = [[f'Puntaje SUS ≥ {META_SUS}', f'{media:.1f} (IC 95 % {li:.1f}–{ls:.1f})', 'Sí' if cumple_sus else 'No']]
if HAY_EV:
    cumple_ms = bool((ms < META_MS).all())
    filas.append([f'Tiempo de respuesta < {META_MS} ms', f'media {ms.mean():.1f} ms; máximo {ms.max():.1f} ms; {100 * (ms < META_MS).mean():.0f} % de las evaluaciones', 'Sí' if cumple_ms else 'No'])
    print('HE4:', 'se cumple completa' if (cumple_sus and cumple_ms) else 'se cumple parcialmente' if (cumple_sus or cumple_ms) else 'no se cumple')
else:
    filas.append([f'Tiempo de respuesta < {META_MS} ms', 'Pendiente (falta el archivo de evaluaciones)', 'Pendiente'])
    print('HE4: el criterio de la SUS', 'se cumple' if cumple_sus else 'no se cumple', '; falta analizar el tiempo de respuesta.')
he4 = pd.DataFrame(filas, columns=['Criterio prefijado', 'Resultado', 'Cumple'])
he4.to_csv(SALIDA / 'he4.csv', index=False)
he4

HE4: el criterio de la SUS se cumple ; falta analizar el tiempo de respuesta.


,Criterio prefijado,Resultado,Cumple
0,Puntaje SUS ≥ 68,93.8 (IC 95 % 89.9–97.8),Sí
1,Tiempo de respuesta < 2000 ms,Pendiente (falta el archivo de evaluaciones),Pendiente
